# MEMC cvičení 4 – Spojitá rozdělení

---

Ve cvičení 3 jsme z jednoho rovnoměrného čísla $\gamma$ vyráběli diskrétní veličiny. Tady uděláme totéž pro **spojitá rozdělení**. Klíčem je jediná věta:

> **Inverzní transformace.** Má-li veličina $X$ spojitou, ostře rostoucí distribuční funkci $F$ a $\gamma \sim R(0, 1)$, pak $X = F^{-1}(\gamma)$ má distribuční funkci $F$.

V sešitu uvidíte:

- proč věta platí a jak podle ní **postupovat krok za krokem**
- generátory: rovnoměrné, exponenciální, **Cauchyho**, trojúhelníkové, po částech konstantní hustota
- proč u Cauchyho rozdělení **nefunguje zákon velkých čísel** (a co se tím mění z pohledu sešitu 0)
- co dělat, když $F^{-1}$ nemá uzavřený vzorec (normální rozdělení)

In [ ]:
import math
import random
import time
from bisect import bisect_right
from itertools import accumulate

import matplotlib.pyplot as plt

random.seed(1)  # opakovatelné výsledky


def samples(sampler, n):
    '''n volání generátoru -> seznam vzorků. Tohle je výstup celé metody.'''
    return [sampler() for _ in range(n)]


def estimate(xs):
    '''Monte Carlo z hotového seznamu: průměr a jeho směrodatná chyba sigma/sqrt(n).'''
    n = len(xs)
    mean = sum(xs) / n
    var = sum((x - mean) ** 2 for x in xs) / (n - 1)
    return mean, math.sqrt(var / n)


def check(name, mean, se, exact, k=5):
    '''Vypíše odhad a ověří, že leží do k směrodatných chyb od přesné hodnoty.'''
    print(f"{name}: {mean:.5f} +- {se:.5f}  (přesně {exact:.5f})")
    assert abs(mean - exact) <= k * se, name


def hist_hustota(ax, xs, span, pdf, nazev, bins=60):
    '''Histogram vzorků (ve výšce odpovídající hustotě) s přesnou hustotou červeně.'''
    w = (span[1] - span[0]) / bins
    ax.hist(xs, bins=bins, range=span, weights=[1 / (len(xs) * w)] * len(xs), alpha=0.6)
    grid = [span[0] + (span[1] - span[0]) * i / 400 for i in range(401)]
    ax.plot(grid, [pdf(x) for x in grid], "r")
    ax.set_title(nazev)

## 1. Inverzní transformace

### Proč to funguje

Distribuční funkce $F$ je rostoucí, takže $F^{-1}$ existuje, a pro $x$ platí

$$P\big(F^{-1}(\gamma) \le x\big) = P\big(\gamma \le F(x)\big) = F(x),$$

protože $\gamma$ je rovnoměrné na $[0, 1]$ a $P(\gamma \le t) = t$. Veličina $F^{-1}(\gamma)$ má tedy přesně distribuční funkci $F$.

### Postup

1. Napíšeme distribuční funkci $F(x) = \int_{-\infty}^x f(t)\,dt$.
2. Vyřešíme rovnici $F(x) = \gamma$ pro $x$ (dostaneme $x = F^{-1}(\gamma)$).
3. Do $F^{-1}$ dosadíme $\gamma = $ `random.random()`.

Je to stejná myšlenka jako u diskrétního případu: tam jsme hledali nejmenší $k$ s $\gamma < F(k)$, tady řešíme $F(x) = \gamma$ přesně.

In [ ]:
# jak inverzní transformace funguje, na příkladu Exp(1): F(x) = 1 - exp(-x)
F_ = lambda x: 1 - math.exp(-x)
Finv_ = lambda gamma: -math.log(1 - gamma)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

ax = axes[0]
xx = [i / 50 for i in range(0, 301)]
ax.plot(xx, [F_(x) for x in xx], "k", lw=2, label="$F(x) = 1 - e^{-x}$")
us = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
for u in us:
    x = Finv_(u)
    ax.annotate("", xy=(x, u), xytext=(0, u), arrowprops=dict(arrowstyle="->", color="tab:blue", alpha=0.6))
    ax.plot([x, x], [0, u], color="tab:red", alpha=0.6)
    ax.plot([x], [0], "o", color="tab:red", clip_on=False)
ax.set_xlabel("$x = F^{-1}(\\gamma)$ (červeně)")
ax.set_ylabel("$\\gamma$ (modře), rovnoměrně rozložené")
ax.set_ylim(0, 1.02)
ax.set_title("Rovnoměrná $\\gamma$ → doprava ke grafu $F$ → dolů na osu $x$")
ax.legend(loc="lower right")

xs_ = [Finv_(1 - random.random()) for _ in range(100_000)]
hist_hustota(axes[1], xs_, (0, 6), lambda x: math.exp(-x), "Výsledek: histogram $F^{-1}(\\gamma)$ a hustota $e^{-x}$")
plt.tight_layout()
plt.show()

Strmá část $F$ (tam, kde je velká hustota) **zhustí** rovnoměrně rozložené hodnoty $\gamma$ na malý úsek osy $x$, a plochá část je naopak rozřídí. Přesně to dělá z rovnoměrného rozdělení rozdělení s hustotou $f = F'$.

### Přehled rozdělení z tohoto sešitu

**Rovnoměrné** $R(a,b)$:

$$F(x) = \frac{x - a}{b - a} \qquad\Longrightarrow\qquad F^{-1}(\gamma) = a + (b - a)\,\gamma$$

**Exponenciální** $\operatorname{Exp}(\lambda)$:

$$F(x) = 1 - e^{-\lambda x} \qquad\Longrightarrow\qquad F^{-1}(\gamma) = -\frac{\ln(1 - \gamma)}{\lambda}$$

**Cauchyho** $C(x_0, s)$:

$$F(x) = \frac{1}{2} + \frac{1}{\pi}\arctan\frac{x - x_0}{s} \qquad\Longrightarrow\qquad F^{-1}(\gamma) = x_0 + s\,\tan\Big(\pi\big(\gamma - \tfrac{1}{2}\big)\Big)$$

**Hustota $2x$ na $[0, 1]$** (výše požadavku z cvičení 2; ve **cvičení 9** s ní pojišťovnu dořešíme):

$$F(x) = x^2 \qquad\Longrightarrow\qquad F^{-1}(\gamma) = \sqrt{\gamma}$$

## 2. Rovnoměrné a exponenciální rozdělení

U exponenciálního rozdělení se vyplatí všimnout si, že $1 - \gamma$ je také rovnoměrné. Můžeme tedy místo $-\ln(1 - \gamma)$ psát i $-\ln \gamma$, ale pozor: `random.random()` vrací číslo z $[0, 1)$, takže $\ln \gamma$ by pro $\gamma = 0$ selhalo. Hodnota $1 - \gamma$ leží v $(0, 1]$ a logaritmus je vždy definovaný.

In [ ]:
def uniform(a, b):
    # F(x) = (x - a)/(b - a)  =>  F^-1(gamma) = a + (b - a)*gamma
    return a + (b - a) * random.random()


def exponential(lam):
    # F(x) = 1 - e^(-lam*x)  =>  F^-1(gamma) = -ln(1 - gamma)/lam
    # 1 - gamma je také rovnoměrné a pro random() z [0, 1) nikdy není 0
    return -math.log(1.0 - random.random()) / lam


n = 200_000

xs = samples(lambda: uniform(2, 5), n)
check("uniform(2, 5): E X", *estimate(xs), 3.5)

xs = samples(lambda: exponential(2.0), n)
check("exponential(2): E X", *estimate(xs), 0.5)

## 3. Cauchyho rozdělení: když zákon velkých čísel selže

Cauchyho rozdělení má hustotu $f(x) = \dfrac{1}{\pi s\,\big(1 + ((x - x_0)/s)^2\big)}$. Vypadá jako zvon, ale s **těžkými chvosty**: velké hodnoty jsou tak časté, že **neexistuje střední hodnota** (integrál $\int |x| f(x)\,dx$ diverguje) a tím spíš ani rozptyl.

Důsledek: nic z toho, co jsme se naučili v sešitu 0, tu neplatí. Zákon velkých čísel, Čebyševova nerovnost ani CLV předpokládají konečnou střední hodnotu a rozptyl.

In [ ]:
def cauchy(x0=0.0, s=1.0):
    # F(x) = 1/2 + atan((x - x0)/s)/pi  =>  F^-1(gamma) = x0 + s*tan(pi*(gamma - 1/2))
    return x0 + s * math.tan(math.pi * (random.random() - 0.5))


def prubezne_prumery(sampler, n):
    s = 0.0
    out = []
    for i in range(1, n + 1):
        s += sampler()
        out.append(s / i)
    return out


fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for _ in range(4):
    axes[0].plot(prubezne_prumery(cauchy, 20_000))
    axes[1].plot(prubezne_prumery(lambda: exponential(1.0), 20_000))
axes[0].set_title("Cauchy: průběžné průměry čtyř běhů")
axes[1].set_title("Exp(1): průběžné průměry čtyř běhů")
axes[1].axhline(1.0, color="black", linestyle="--")
for ax in axes:
    ax.set_xlabel("počet vzorků N")
plt.tight_layout()
plt.show()

**Pozor, jeden běh klame.** Křivky u Cauchyho se po čase mohou zdát „klidné": každý nový vzorek $x$ změní průměr jen o $x/N$, takže se průběh zpomaluje. Jenže kdykoli přijde extrémní hodnota, průměr znovu skočí (u jednoho z běhů je to vidět), a **kam se průměr ustálí, je u každého běhu jinak**. U exponenciálního rozdělení se všechny běhy stahují ke stejné hodnotě $1$.

Rozhodující je tedy to, jak jsou průměry **rozložené napříč mnoha běhy**. Pro každé $N$ vezmeme 300 nezávislých běhů a zobrazíme dolní a horní kvartil (mezi nimi leží polovina běhů) odchylky průměru od středu:

In [ ]:
kontrolni_body = [10, 30, 100, 300, 1_000, 3_000, 10_000]
body = set(kontrolni_body)


def kvartily_prumeru(sampler, stred, behy=300):
    '''Pro každé N z kontrolních bodů: dolní a horní kvartil hodnoty (průměr z N vzorků - stred) napříč běhy.'''
    hodnoty = {N: [] for N in kontrolni_body}
    for _ in range(behy):
        s = 0.0
        for i in range(1, kontrolni_body[-1] + 1):
            s += sampler()
            if i in body:
                hodnoty[i].append(s / i - stred)
    dolni, horni = [], []
    for N in kontrolni_body:
        h = sorted(hodnoty[N])
        dolni.append(h[len(h) // 4])
        horni.append(h[3 * len(h) // 4])
    return dolni, horni


plt.figure(figsize=(7.5, 4.5))
for nazev, sampler, stred, barva in (("Cauchy (střed 0)", cauchy, 0.0, "tab:red"),
                                      ("Exp(1) (střed 1)", lambda: exponential(1.0), 1.0, "tab:blue")):
    dolni, horni = kvartily_prumeru(sampler, stred)
    plt.fill_between(kontrolni_body, dolni, horni, alpha=0.3, color=barva, label=nazev)
    plt.plot(kontrolni_body, dolni, color=barva)
    plt.plot(kontrolni_body, horni, color=barva)
plt.xscale("log")
plt.xlabel("počet vzorků N v průměru")
plt.ylabel("průměr minus střed")
plt.title("Polovina běhů leží uvnitř pásu (kvartily napříč 300 běhy)")
plt.legend()
plt.show()

U exponenciálního rozdělení se pás zužuje jako $1/\sqrt{N}$, a průměry tedy s rostoucím $N$ konvergují. U Cauchyho zůstává pás **pořád stejně široký**, kvartily jsou (až na šum z 300 běhů) na $\pm 1$: to jsou kvartily samotného Cauchyho rozdělení, protože $P(|X| < 1) = \tfrac12$. Průměr z $N$ nezávislých Cauchyho veličin má **opět přesně stejné Cauchyho rozdělení**, ať je $N$ jakkoli velké. Průměrování tu nepomáhá vůbec.

### Co se **dá** odhadnout

Střední hodnota neexistuje, ale pravděpodobnosti existují vždycky. Z těch samých vzorků odhadneme třeba $P(|X| < 1) = \tfrac12$ (průměr indikátorů je Bernoulliho veličina s konečným rozptylem):

In [ ]:
xs = samples(cauchy, n)
hits = [float(abs(x) < 1) for x in xs]
check("P(|Cauchy| < 1)", *estimate(hits), 0.5)

# průměr 100 Cauchyho veličin má stejné rozdělení jako jedna:
prumery = [sum(cauchy() for _ in range(100)) / 100 for _ in range(5_000)]
check("P(|průměr ze 100| < 1)", *estimate([float(abs(m) < 1) for m in prumery]), 0.5)

Druhý řádek ukazuje to nejpodstatnější: pravděpodobnost $|\text{průměr}| < 1$ je $0.5$ stejně pro $N = 1$ i pro $N = 100$. U rozdělení s konečným rozptylem by se průměry s rostoucím $N$ stahovaly k $\mu$ a tato pravděpodobnost by rostla k $1$.

## 4. Trojúhelníkové rozdělení: $F$ po částech

Hustota roste lineárně na $[a, c]$ a klesá lineárně na $[c, b]$. Distribuční funkce je **po částech** kvadratická, a proto i inverzní funkce má dva vzorce. Hodnotu $F(c) = \dfrac{c - a}{b - a}$ použijeme jako hranici, kterou porovnáme s $\gamma$:

$$F(x) = \begin{cases}
\dfrac{(x-a)^2}{(b-a)(c-a)}, & a \le x \le c,\\[2mm]
1 - \dfrac{(b-x)^2}{(b-a)(b-c)}, & c < x \le b,
\end{cases}
\qquad
F^{-1}(\gamma) = \begin{cases}
a + \sqrt{\gamma\,(b-a)(c-a)}, & \gamma < F(c),\\[1mm]
b - \sqrt{(1-\gamma)(b-a)(b-c)}, & \gamma \ge F(c).
\end{cases}$$

In [ ]:
def triangular(a, b, c):
    '''Hustota roste na [a, c] a klesá na [c, b]. Inverzní funkce je definovaná po částech.'''
    gamma = random.random()
    hranice = (c - a) / (b - a)   # F(c)
    if gamma < hranice:
        return a + math.sqrt(gamma * (b - a) * (c - a))
    return b - math.sqrt((1.0 - gamma) * (b - a) * (b - c))


xs = samples(lambda: triangular(0, 4, 1), n)
check("triangular(0, 4, 1): E X", *estimate(xs), (0 + 4 + 1) / 3)

## 5. Po částech konstantní hustota

Hustota je konstantní na úsecích $[e_i, e_{i+1})$ s výškami $h_i$ (schodovitá hustota). Distribuční funkce je po částech lineární a inverzní funkci tedy nemá jediný vzorec. Naštěstí se úloha **rozdělí na dvě jednoduché**:

1. **Který úsek?** Je to diskrétní úloha: úsek $i$ má pravděpodobnost rovnou svému obsahu $p_i = \dfrac{(e_{i+1} - e_i)\,h_i}{\sum_j (e_{j+1} - e_j)\,h_j}$ (základní algoritmus ze cvičení 3).
2. **Kde v úseku?** Uvnitř úseku je hustota konstantní, takže použijeme rovnoměrné rozdělení.

Výšky $h_i$ nemusí být normované, normalizaci obstará dělení součtem. Začneme s pěti úseky:

In [ ]:
def piecewise_const(hranice, vysky):
    '''Po částech konstantní hustota.

    hranice = [e_0, e_1, ..., e_n]   hranice úseků, o jednu víc než výšek
    vysky   = [h_0, ..., h_(n-1)]    výška na úseku [e_i, e_(i+1)); nemusí být normovaná
    '''
    n_useku = len(vysky)

    # obsah každého úseku = šířka * výška; po dělení součtem je to pravděpodobnost p_i
    obsahy = [(hranice[i + 1] - hranice[i]) * vysky[i] for i in range(n_useku)]
    celkem = sum(obsahy)

    # krok 1: odečítej p_i, dokud nespadneš pod nulu (základní algoritmus z diskrétního případu)
    gamma = random.random()
    for i in range(n_useku):
        gamma -= obsahy[i] / celkem
        if gamma < 0:
            break
    # po skončení cyklu drží i index vybraného úseku
    # (kdyby zaokrouhlení nechalo gamma >= 0, zůstane i = n_useku - 1, což je správná záloha)

    # krok 2: uvnitř úseku je hustota konstantní
    return uniform(hranice[i], hranice[i + 1])


def stredni_hodnota_schodu(hranice, vysky):
    '''Přesná střední hodnota schodovité hustoty: součet (pravděpodobnost úseku) * (střed úseku).'''
    obsahy = [(hranice[i + 1] - hranice[i]) * vysky[i] for i in range(len(vysky))]
    stredy = [(hranice[i] + hranice[i + 1]) / 2 for i in range(len(vysky))]
    return sum(o * s for o, s in zip(obsahy, stredy)) / sum(obsahy)


# pět úseků šířky 1, nenormované výšky
hr5 = [0, 1, 2, 3, 4, 5]
vy5 = [1, 3, 2, 0.5, 1.5]

xs = samples(lambda: piecewise_const(hr5, vy5), n)
check("piecewise (5 úseků): E X", *estimate(xs), stredni_hodnota_schodu(hr5, vy5))

### Schody aproximují libovolnou křivku

Skutečná síla této metody je jinde: **každou hustotu $f$ lze nahradit schodovitou**. Interval $[a, b]$ rozdělíme na $n$ malých úseků šířky $\Delta x$ a na každém nahradíme $f$ konstantou (například hodnotou uprostřed úseku). Plocha sloupce $f(x_i)\,\Delta x$ je přibližně pravděpodobnost, že hodnota padne do úseku $i$. Čím jemnější dělení, tím lepší aproximace, a protože výšky nemusí být normované, nemusíme ani počítat normalizační konstantu.

Zkusme to na „náhodné" křivce se dvěma kopci, pro kterou nemá $F^{-1}$ pěkný vzorec:

In [ ]:
def krivka(x):
    '''Libovolná (nenormovaná) hustota na [0, 4]: dva kopce.'''
    return math.exp(-((x - 1) / 0.4) ** 2) + 0.6 * math.exp(-((x - 3) / 0.7) ** 2)


def schody(f, a, b, n_useku):
    '''Rozdělí [a, b] na n_useku úseků; výška = hodnota f uprostřed úseku.'''
    dx = (b - a) / n_useku
    hranice = [a + i * dx for i in range(n_useku + 1)]
    vysky = [f(a + (i + 0.5) * dx) for i in range(n_useku)]
    return hranice, vysky


def make_bisect(p):
    '''Diskrétní rozdělení s pravděpodobnostmi p: kumulativní součty + půlení intervalu (cvičení 3).'''
    cum = list(accumulate(p))
    return lambda: min(bisect_right(cum, random.random()), len(p) - 1)


def make_piecewise_const(hranice, vysky):
    '''Totéž co piecewise_const, ale úsek se hledá půlením intervalu: O(log n) místo O(n) na losování.'''
    obsahy = [(hranice[i + 1] - hranice[i]) * vysky[i] for i in range(len(vysky))]
    celkem = sum(obsahy)
    vyber = make_bisect([o / celkem for o in obsahy])

    def sample():
        i = vyber()
        return uniform(hranice[i], hranice[i + 1])

    return sample


# "přesná" hodnota: velmi jemné schody (100 000 úseků) berou za pravdu
hr_ref, vy_ref = schody(krivka, 0, 4, 100_000)
E_presne = stredni_hodnota_schodu(hr_ref, vy_ref)
Z = sum(vy_ref) * (4 / 100_000)   # plocha pod křivkou (normalizační konstanta)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
grid = [4 * i / 400 for i in range(401)]
for ax, n_useku in zip(axes, (4, 10, 40)):
    hr, vy = schody(krivka, 0, 4, n_useku)
    sampler = make_piecewise_const(hr, vy)
    xs = samples(sampler, 100_000)
    w = 4 / 40
    ax.hist(xs, bins=40, range=(0, 4), weights=[1 / (len(xs) * w)] * len(xs), alpha=0.5, label="vzorky")
    ax.plot(grid, [krivka(x) / Z for x in grid], "r", label="skutečná hustota")
    A = sum(vy) * (4 / n_useku)   # plocha schodů (liší se od Z; vzorkovač normuje podle ní)
    ax.step(hr, [v / A for v in vy] + [vy[-1] / A], where="post", color="green", label="schody (normované)")
    ax.set_title(f"{n_useku} {'úseky' if n_useku < 5 else 'úseků'}")
axes[0].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

Už se čtyřmi úseky je vidět, o co jde, a s čtyřiceti se schody od křivky téměř neodlišují. Jak rychle se aproximace zlepšuje? Střední hodnota schodovité hustoty se dá spočítat **přesně** ze vzorce nahoře, takže chybu aproximace vidíme bez šumu simulace:

In [ ]:
print(f"přesná střední hodnota křivky: {E_presne:.6f}\n")
print(f"{'úseků':>7s} {'E X (schody)':>14s} {'chyba':>12s}")
predchozi = None
for n_useku in (2, 4, 8, 16, 32, 64, 128):
    hr, vy = schody(krivka, 0, 4, n_useku)
    E = stredni_hodnota_schodu(hr, vy)
    chyba = abs(E - E_presne)
    pomer = f"   (předchozí chyba / tato = {predchozi / chyba:.1f})" if predchozi else ""
    print(f"{n_useku:7d} {E:14.6f} {chyba:12.2e}{pomer}")
    predchozi = chyba

# a simulace ze schodů s 64 úseky odpovídá jejich přesné střední hodnotě
hr, vy = schody(krivka, 0, 4, 64)
check("simulace ze 64 schodů: E X", *estimate(samples(make_piecewise_const(hr, vy), n)), stredni_hodnota_schodu(hr, vy))

Pro hrubé dělení (do osmi úseků) se chyba chová nepravidelně, protože úseky jsou širší než kopce křivky. Jakmile jsou ale úseky dost úzké, **zdvojnásobení počtu úseků chybu přesně zčtvrtí** (poměr $4.0$ v tabulce): jde o aproximaci druhého řádu, stejně jako u obdélníkového (středového) pravidla v kvadratuře. Pro velké $n$ se vyplatí hledat úsek půlením intervalu (`make_piecewise_const`): stojí $O(\log n)$ místo $O(n)$ na losování.

> **Poznámka:** schodovitá hustota je **jiná** hustota než původní. Aproximaci jsme tím neodstranili, jen jsme ji vyměnili za chybu, kterou umíme řídit počtem úseků. Pokud chceme vzorky z přesně té původní hustoty, použijeme zamítací metodu (cvičení 6).

## 6. Všechna rozdělení najednou

Pro kontrolu srovnáme histogramy z našich generátorů s přesnými hustotami.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

xs = samples(lambda: uniform(2, 5), n)
hist_hustota(axes[0][0], xs, (2, 5), lambda x: 1 / 3, "Rovnoměrné(2, 5)")

xs = samples(lambda: exponential(2.0), n)
hist_hustota(axes[0][1], xs, (0, 3), lambda x: 2 * math.exp(-2 * x), "Exponenciální(2)")

xs = samples(cauchy, n)
hist_hustota(axes[0][2], xs, (-4, 4), lambda x: 1 / (math.pi * (1 + x * x)), "Cauchy(0, 1), výřez (-4, 4)")


def tri_pdf(a, b, c):
    def f(x):
        if x < a or x > b:
            return 0.0
        if x <= c:
            return 2 * (x - a) / ((b - a) * (c - a))
        return 2 * (b - x) / ((b - a) * (b - c))
    return f


xs = samples(lambda: triangular(0, 4, 1), n)
hist_hustota(axes[1][0], xs, (0, 4), tri_pdf(0, 4, 1), "Trojúhelníkové(0, 4, 1)")

xs = samples(lambda: piecewise_const(hr5, vy5), n)


def schod_pdf(hranice, vysky):
    celkem = sum((hranice[i + 1] - hranice[i]) * vysky[i] for i in range(len(vysky)))

    def f(x):
        for i in range(len(vysky)):
            if hranice[i] <= x < hranice[i + 1]:
                return vysky[i] / celkem
        return 0.0
    return f


hist_hustota(axes[1][1], xs, (0, 5), schod_pdf(hr5, vy5), "Po částech konstantní (5 úseků)", bins=50)

xs = samples(lambda: math.sqrt(random.random()), n)
hist_hustota(axes[1][2], xs, (0, 1), lambda x: 2 * x, "Hustota 2x (výše požadavku, cvičení 2)")

plt.tight_layout()
plt.show()

Histogramy (modře) kopírují hustoty (červeně). U Cauchyho rozdělení kreslíme jen výřez $(-4, 4)$, protože chvosty sahají do nekonečna.

## 7. Když $F^{-1}$ nemá vzorec: numerická inverze

Normální rozdělení má $F(x) = \Phi(x)$, kterou umíme spočítat (přes chybovou funkci `erf`), ale rovnici $\Phi(x) = \gamma$ nelze vyřešit uzavřeným vzorcem. Řešení: **vyřešit ji numericky půlením intervalu**, protože $F$ je rostoucí. Postup je obecný, funguje pro libovolnou spojitou rostoucí $F$.

Cena je ale vysoká: každé losování znamená desítky výpočtů $F$. Lepší metody pro normální rozdělení (zamítací metoda) uvidíte ve cvičení 6.

In [ ]:
def Phi(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))


def inverze_pulenim(F, gamma, lo=-10.0, hi=10.0, iteraci=60):
    '''Řeší F(x) = gamma půlením intervalu [lo, hi]; F musí být rostoucí.'''
    for _ in range(iteraci):
        mid = (lo + hi) / 2
        if F(mid) < gamma:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


m_norm = 50_000
t0 = time.perf_counter()
xs = samples(lambda: inverze_pulenim(Phi, random.random()), m_norm)
t = (time.perf_counter() - t0) / m_norm

check("N(0,1): E X", *estimate(xs), 0.0)
check("N(0,1): E X^2", *estimate([x * x for x in xs]), 1.0)
print(f"čas na jedno losování: {t * 1e6:.0f} mikrosekund (60 výpočtů F)")

## 8. Alternativa: NumPy

NumPy má všechna tato rozdělení hotová. Pozor na **parametrizaci**: `exponential` bere `scale` $= 1/\lambda$ (ne $\lambda$), `triangular` bere `left, mode, right`.

In [ ]:
try:
    import numpy as np

    rng = np.random.default_rng(1)
    N = 1_000_000

    t0 = time.perf_counter()
    u = rng.uniform(2, 5, size=N)
    e = rng.exponential(1 / 2.0, size=N)            # scale = 1/lambda
    c = rng.standard_cauchy(size=N)
    tr = rng.triangular(0, 1, 4, size=N)             # left, mode, right (pořadí je jiné než u nás!)
    t = time.perf_counter() - t0

    print(f"E uniform(2,5) = {u.mean():.4f}  (3.5)")
    print(f"E exponential(2) = {e.mean():.4f}  (0.5)")
    print(f"P(|Cauchy| < 1) = {(np.abs(c) < 1).mean():.4f}  (0.5)")
    print(f"E triangular(0,4,1) = {tr.mean():.4f}  ({5 / 3:.4f})")
    print(f"{4 * N:,} vzorků za {t:.2f} s".replace(",", " "))

    # inverzní transformace vektorově: celé pole najednou
    gamma = rng.random(N)
    print(f"inverzní transformace, exponenciální: {(-np.log(1 - gamma) / 2.0).mean():.4f}  (0.5)")
except ModuleNotFoundError:
    print("NumPy není nainstalované, spusťte:  uv sync")

Všimněte si pořadí parametrů u `triangular`: NumPy bere `(left, mode, right)`, náš generátor `(a, b, c)` s vrcholem $c$ jako **posledním**. Při přepisu kódu mezi knihovnami je to častá past.

## Shrnutí

| Pojem | Co si pamatovat |
|-------|-----------------|
| Inverzní transformace | $X = F^{-1}(\gamma)$ má distribuční funkci $F$; stačí vyřešit $F(x) = \gamma$ |
| Exponenciální | $-\ln(1 - \gamma)/\lambda$; použijeme $1 - \gamma$, protože `random()` může vrátit $0$ |
| Po částech definované $F$ | inverzní vzorec po částech (trojúhelníkové), porovnáváme $\gamma$ s $F$ v hranici |
| Po částech konstantní hustota | dvě úlohy: úsek (diskrétně), pak rovnoměrně uvnitř úseku; s dostatkem úseků aproximuje libovolnou hustotu (od dost jemného dělení chyba klesá jako $1/n^2$) |
| Cauchyho | nemá střední hodnotu, zákon velkých čísel ani CLV neplatí; pravděpodobnosti odhadnout jde |
| $F^{-1}$ bez vzorce | numerická inverze půlením intervalu, obecná, ale pomalá |
| Testování | střední hodnota, případně $E X^2$ nebo pravděpodobnost; histogram proti hustotě |

### Zkuste si

1. **Paretovo rozdělení** s $F(x) = 1 - x^{-\alpha}$ pro $x \ge 1$: odvoďte $F^{-1}$, napište generátor a ověřte $E X = \dfrac{\alpha}{\alpha - 1}$ pro $\alpha = 3$ (pro $\alpha \le 1$ střední hodnota neexistuje; co se stane s odhadem?).
2. **Laplaceovo rozdělení** (dvojitá exponenciála) s hustotou $f(x) = \tfrac12 e^{-|x|}$: $F$ je po částech definovaná. Odvoďte $F^{-1}$ a ověřte $E X = 0$, $E X^2 = 2$.
3. Rozdělení **maxima dvou rovnoměrných** $\max(\gamma_1, \gamma_2)$ má $F(x) = x^2$. Ukažte, že to je totéž rozdělení jako $\sqrt{\gamma}$, a ověřte to simulací dvěma způsoby.